# <font color="#418FDE" size="6.5" uppercase>**Scaling Distributions**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Apply standard, min-max, max-absolute, robust, and sample normalization transformations. 
- Compare quantile and power transformations for skewed numerical features. 
- Choose scaling strategies that respect sparse and dense data constraints. 


## **1. Core Scaling Methods**

### **1.1. Standard Scaling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_01_01.jpg?v=1783905423" width="250">



>* Centers features around mean and spread
>* Makes different feature scales comparable

>* Balances distance and gradient based algorithms
>* Prevents large-scale features dominating PCA

>* Outliers can distort standard scaling results
>* Fit scaling only on training data



In [ ]:
#@title Python Code - Standard Scaling

# Demonstrate standard scaling with simple numeric features.
# Compare original units against standardized feature values.
# Keep output short for beginner friendly learning.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create tiny data with different measurement units.
data = pd.DataFrame({
    "height_cm": [150, 160, 170, 180, 190],
    "income_dollars": [30000, 45000, 60000, 90000, 120000]
})

# Validate the table before numeric calculations.
assert data.shape == (5, 2)
assert data.notna().all().all()

# Compute feature means and population standard deviations.
means = data.mean(axis=0)
stds = data.std(axis=0, ddof=0)

# Apply the standard scaling formula manually.
scaled = (data - means) / stds
scaled = scaled.round(2)

# Print compact summaries for both feature spaces.
print("Original means:", means.round(2).to_dict())
print("Original stds:", stds.round(2).to_dict())
print("Scaled means:", scaled.mean().round(2).to_dict())
print("Scaled stds:", scaled.std(ddof=0).round(2).to_dict())

# Plot original and scaled values side by side.
fig, axes = plt.subplots(1, 2, figsize=(9, 3))

# Show how raw magnitudes differ strongly.
axes[0].plot(data.index, data["height_cm"], marker="o")
axes[0].plot(data.index, data["income_dollars"], marker="o")
axes[0].set_title("Original values")

# Show standardized values on comparable scales.
axes[1].plot(scaled.index, scaled["height_cm"], marker="o")
axes[1].plot(scaled.index, scaled["income_dollars"], marker="o")
axes[1].set_title("Standard scaled values")

# Add labels and legends for interpretation.
for axis in axes:
    axis.set_xlabel("Customer index")
    axis.legend(loc="best")

plt.tight_layout()
plt.show()



### **1.2. Min Max Scaling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_01_02.jpg?v=1783905421" width="250">



>* Rescales values to a fixed range
>* Preserves order while making features comparable

>* Preserves distribution shape within fixed bounds
>* Helps algorithms compare differently scaled features

>* Outliers can compress typical scaled values
>* Inspect data; consider robust scaling for tails



In [ ]:
#@title Python Code - Min Max Scaling

# Demonstrate min max scaling manually.
# Compare original and scaled feature ranges.
# Keep output short for beginners.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create tiny dense numerical data.
data = pd.DataFrame({
    "size_sqft": [650, 900, 1200, 1800],
    "income_usd": [42000, 58000, 76000, 110000],
    "commute_min": [12, 25, 40, 55],
})

# Validate the table before scaling.
assert data.shape == (4, 3)
assert data.notna().all().all()

# Compute column minimums and maximums.
mins = data.min(axis=0)
maxs = data.max(axis=0)
ranges = maxs - mins

# Prevent division by zero safely.
assert (ranges > 0).all()
scaled = (data - mins) / ranges

# Show compact before and after summaries.
summary = pd.DataFrame({
    "original_min": mins,
    "original_max": maxs,
    "scaled_min": scaled.min(axis=0),
    "scaled_max": scaled.max(axis=0),
})

# Print only a small rounded summary.
print("Min max scaling maps each feature into zero through one.")
print(summary.round(2).to_string())

# Plot one feature before and after scaling.
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].bar(data.index, data["income_usd"], color="steelblue")
axes[1].bar(scaled.index, scaled["income_usd"], color="seagreen")

# Label plots with concise titles.
axes[0].set_title("Original income")
axes[1].set_title("Min max scaled income")
axes[0].set_ylabel("US dollars")
axes[1].set_ylabel("Scaled value")

# Keep the plot readable.
for axis in axes:
    axis.set_xlabel("Observation")
    axis.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()



### **1.3. Max Absolute Scaling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_01_03.jpg?v=1783905425" width="250">



>* Scales values by largest absolute feature value
>* Preserves signs without centering or shifting

>* Keeps zero values unchanged in sparse data
>* Reduces memory and computation for large matrices

>* Extreme values can compress typical observations
>* Best when maxima are meaningful



In [ ]:
#@title Python Code - Max Absolute Scaling

# Demonstrate max absolute scaling clearly.
# Preserve signs and zero values.
# Compare original and scaled features.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create small signed feature data.
values = np.array([-8, -4, 0, 2, 10], dtype=float)
feature_name = "daily_return_percent"

# Validate the example before scaling.
assert values.size > 0 and np.isfinite(values).all()
max_abs_value = np.max(np.abs(values))

# Scale by the largest absolute value.
scaled_values = values / max_abs_value
result = pd.DataFrame({feature_name: values, "max_abs_scaled": scaled_values})

# Print a compact comparison table.
print("Max absolute value:", max_abs_value)
print(result.to_string(index=False))

# Check important max absolute properties.
print("Scaled range:", scaled_values.min(), "to", scaled_values.max())
print("Zero stays zero:", scaled_values[values == 0][0] == 0)

# Plot original and scaled values together.
plt.figure(figsize=(7, 4))
plt.plot(values, marker="o", label="Original values")
plt.plot(scaled_values, marker="o", label="Max absolute scaled")
plt.axhline(0, color="black", linewidth=1)
plt.title("Max Absolute Scaling Preserves Sign")
plt.xlabel("Observation index")
plt.ylabel("Value")
plt.legend()
plt.tight_layout()
plt.show()



## **2. Robust Transforming**

### **2.1. Robust Scaling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_02_01.jpg?v=1783905409" width="250">



>* Scales skewed features using median-based summaries
>* Limits outlier influence on typical values

>* Robust scaling preserves skewed feature shape
>* Quantile and power transforms reshape distributions

>* Scale skewed features while preserving structure
>* Use transformations for extreme skewness



In [ ]:
#@title Python Code - Robust Scaling

# Robust scaling resists extreme numerical values.
# Median and IQR define the transformation.
# This example compares common scaling choices.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a small skewed purchase dataset.
purchases = np.array([12, 15, 18, 20, 22, 25, 30, 35, 45, 60, 500], dtype=float)

# Validate the feature before transforming.
assert purchases.ndim == 1 and purchases.size >= 5

# Compute standard scaling using mean and deviation.
standard_scaled = (purchases - purchases.mean()) / purchases.std()

# Compute robust scaling using median and IQR.
median_value = np.median(purchases)
q1_value, q3_value = np.percentile(purchases, [25, 75])

# Protect against division by zero.
iqr_value = max(q3_value - q1_value, 1e-12)
robust_scaled = (purchases - median_value) / iqr_value

# Summarize how the outlier affects each method.
summary = pd.DataFrame({
    "original": purchases,
    "standard": standard_scaled,
    "robust": robust_scaled,
})

# Print only a compact teaching summary.
print("Original median and mean:", round(median_value, 2), round(purchases.mean(), 2))
print("Original IQR and standard deviation:", round(iqr_value, 2), round(purchases.std(), 2))
print("Largest value after standard scaling:", round(standard_scaled[-1], 2))
print("Largest value after robust scaling:", round(robust_scaled[-1], 2))
print("Middle purchase after robust scaling:", round(robust_scaled[5], 2))

# Plot original, standard, and robust values.
fig, axes = plt.subplots(1, 3, figsize=(11, 3))

# Use shared index positions for comparison.
positions = np.arange(purchases.size)

# Draw compact scatter plots for each scale.
axes[0].scatter(positions, summary["original"], color="steelblue")
axes[1].scatter(positions, summary["standard"], color="darkorange")
axes[2].scatter(positions, summary["robust"], color="seagreen")

# Add short titles to each subplot.
axes[0].set_title("Original dollars")
axes[1].set_title("Standard scaled")
axes[2].set_title("Robust scaled")

# Label axes without printing large tables.
for axis in axes:
    axis.set_xlabel("Customer index")
    axis.grid(True, alpha=0.3)

# Keep the figure readable in Colab.
plt.tight_layout()
plt.show()



### **2.2. Sample Normalization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_02_02.jpg?v=1783905411" width="250">



>* Scales each row, not each feature
>* Highlights patterns over total sample size

>* Quantile and power reshape skewed features
>* Sample normalization compares relative row patterns

>* Keep magnitude when it carries signal
>* Normalize when comparing relative sample patterns



In [ ]:
#@title Python Code - Sample Normalization

# Sample normalization compares rows, not columns.
# This example uses tiny document counts.
# Row magnitude can hide useful patterns.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create three short word count samples.
columns = ["sports", "finance", "health"]
counts = np.array([[90, 9, 1], [900, 90, 10], [5, 45, 50]], dtype=float)

# Validate the tiny matrix before transforming.
assert counts.ndim == 2 and counts.shape == (3, 3)
row_totals = counts.sum(axis=1, keepdims=True)
assert np.all(row_totals > 0)

# Normalize each sample to sum one.
normalized = counts / row_totals
raw_frame = pd.DataFrame(counts, columns=columns)
norm_frame = pd.DataFrame(normalized, columns=columns)

# Summarize row totals before normalization.
print("Raw row totals:", row_totals.ravel().astype(int).tolist())
print("Normalized row totals:", normalized.sum(axis=1).round(2).tolist())

# Compare two rows with identical proportions.
print("Rows one and two match after normalization:", np.allclose(normalized[0], normalized[1]))
print("Row three has a different internal pattern.")

# Plot normalized profiles for visual comparison.
ax = norm_frame.plot(kind="bar", figsize=(7, 4))
ax.set_title("Sample normalization emphasizes within-row proportions")
ax.set_xlabel("Sample")
ax.set_ylabel("Proportion within sample")

# Keep the single plot compact and readable.
ax.legend(title="Feature", loc="upper right")
plt.tight_layout()
plt.show()



### **2.3. Sparse Safe Scaling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_02_03.jpg?v=1783905413" width="250">



>* Preserve meaningful zeros in sparse data
>* Avoid transformations that make data dense

>* Quantile transforms reshape skew but challenge zeros
>* Power transforms can preserve sparse absence

>* Preserve meaningful zeros in sparse data
>* Prefer transformations that keep sparsity intact



In [ ]:
#@title Python Code - Sparse Safe Scaling

# Sparse scaling protects meaningful zero entries.
# Dense centering can destroy sparse storage.
# Nonzero-only transforms keep absence interpretable.

import numpy as np
import matplotlib.pyplot as plt

# Create tiny sparse-like count data.
counts = np.array([
    [0, 0, 3, 0],
    [0, 5, 0, 0],
    [2, 0, 0, 9],
    [0, 0, 0, 0],
])

# Count nonzero entries before scaling.
original_nonzero = np.count_nonzero(counts)
original_total = counts.size

# Mean centering changes every structural zero.
centered = counts - counts.mean(axis=0)
centered_nonzero = np.count_nonzero(centered)

# Max-absolute scaling preserves all zero entries.
max_abs = np.maximum(np.abs(counts).max(axis=0), 1)
maxabs_scaled = counts / max_abs
maxabs_nonzero = np.count_nonzero(maxabs_scaled)

# Log scaling compresses positive skew safely.
log_scaled = np.where(counts > 0, np.log1p(counts), 0)
log_nonzero = np.count_nonzero(log_scaled)

# Print a compact sparsity comparison.
print("Original nonzeros:", original_nonzero, "of", original_total)
print("Mean-centered nonzeros:", centered_nonzero, "of", original_total)
print("Max-abs nonzeros:", maxabs_nonzero, "of", original_total)
print("Log1p nonzeros:", log_nonzero, "of", original_total)
print("Lesson: avoid centering when zeros must stay zero.")

# Plot original and sparse-safe transformed values.
labels = ["original", "centered", "maxabs", "log1p"]
nonzeros = [original_nonzero, centered_nonzero, maxabs_nonzero, log_nonzero]

# Show how transformations affect sparsity.
plt.figure(figsize=(7, 4))
plt.bar(labels, nonzeros, color=["gray", "tomato", "seagreen", "royalblue"])
plt.ylabel("Stored nonzero values")
plt.title("Sparse Safe Scaling Keeps Zeros Sparse")
plt.ylim(0, original_total)
plt.show()



## **3. Distribution Shape**

### **3.1. Quantile Mapping**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_03_01.jpg?v=1783905415" width="250">



>* Ranks values to reshape uneven distributions
>* Spreads data while preserving observation order

>* Dense features often suit quantile mapping
>* Sparse zeros may lose meaning and efficiency

>* Use quantile mapping for dense skewed features
>* Preserve sparsity and fit on training data



In [ ]:
#@title Python Code - Quantile Mapping

# Quantile mapping reshapes ranked numerical values.
# Dense features can safely change distribution.
# Sparse zeros need special caution.

import numpy as np
import matplotlib.pyplot as plt

# Create a tiny skewed dense feature.
rng = np.random.default_rng(7)
amounts = rng.lognormal(mean=2.0, sigma=1.0, size=80)

# Add a few extreme purchase totals.
amounts[-3:] = np.array([80.0, 120.0, 200.0])
assert amounts.size == 80 and np.all(amounts > 0)

# Compute rank based uniform quantile scores.
order = np.argsort(amounts)
ranks = np.empty_like(order, dtype=float)
ranks[order] = np.arange(1, amounts.size + 1)

# Map ranks into the open interval.
quantiles = (ranks - 0.5) / amounts.size
assert quantiles.min() > 0 and quantiles.max() < 1

# Show sparse zero behavior conceptually.
sparse_like = np.array([0, 0, 0, 0, 3, 8, 20], dtype=float)
zero_count = int(np.sum(sparse_like == 0))

# Print only compact teaching summaries.
print("Dense skewed feature: purchase totals in dollars.")
print(f"Original range: {amounts.min():.2f} to {amounts.max():.2f}")
print(f"Quantile range: {quantiles.min():.3f} to {quantiles.max():.3f}")
print(f"Sparse example has {zero_count} zeros before mapping.")
print("Quantile mapping may turn many sparse zeros nonzero.")

# Compare original and quantile mapped shapes.
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].hist(amounts, bins=15, color="steelblue", edgecolor="white")
axes[1].hist(quantiles, bins=15, color="seagreen", edgecolor="white")

# Label plots for beginner interpretation.
axes[0].set_title("Original skewed values")
axes[1].set_title("Uniform quantile mapping")
axes[0].set_xlabel("Dollars")
axes[1].set_xlabel("Rank-based quantile")

# Keep the single plot compact.
axes[0].set_ylabel("Count")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()



### **3.2. Power Transform**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_03_02.jpg?v=1783905417" width="250">



>* Makes skewed features more balanced
>* Reduces extreme values before scaling

>* Match transforms to value ranges
>* Use cautiously with sparse zero-heavy data

>* Preserve order while shrinking extreme gaps
>* Use when performance outweighs original-scale interpretability



In [ ]:
#@title Python Code - Power Transform

# Demonstrate power transforms on skewed dense data.
# Compare positive and signed feature transformations.
# Highlight sparse data caution with zeros.

import numpy as np
import matplotlib.pyplot as plt

# Create deterministic skewed dense measurements.
rng = np.random.default_rng(7)
spending = rng.lognormal(mean=3.0, sigma=0.9, size=120)

# Create signed changes around a baseline.
changes = rng.normal(loc=0.0, scale=18.0, size=120)
changes[:8] = changes[:8] * 4.0

# Validate small one dimensional arrays.
assert spending.ndim == 1 and spending.size == 120
assert changes.ndim == 1 and changes.size == 120

# Apply log transform for positive values.
log_spending = np.log1p(spending)

# Apply signed power transform manually.
signed_power = np.sign(changes) * np.sqrt(np.abs(changes))

# Summarize spread before and after transforming.
raw_ratio = np.percentile(spending, 95) / np.percentile(spending, 50)
log_ratio = np.percentile(log_spending, 95) / np.percentile(log_spending, 50)

# Count zeros in a sparse-like feature.
sparse_like = np.array([0, 0, 0, 0, 2, 0, 5, 0, 12, 0])
zero_count = np.count_nonzero(sparse_like == 0)

# Show concise teaching messages.
print(f"Positive feature: 95th/median ratio {raw_ratio:.2f} -> {log_ratio:.2f}")
print("Signed feature: square-root power keeps negatives negative.")
print(f"Sparse-like zeros preserved here: {zero_count} of {sparse_like.size}")
print("Use power transforms mainly on dense numerical features.")

# Plot original and transformed positive distributions.
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].hist(spending, bins=18, color="tomato", edgecolor="white")
axes[1].hist(log_spending, bins=18, color="seagreen", edgecolor="white")

# Add clear titles and labels.
axes[0].set_title("Original spending")
axes[1].set_title("Log power transform")
axes[0].set_xlabel("Dollars")
axes[1].set_xlabel("Transformed scale")

# Keep the plot compact.
for axis in axes:
    axis.set_ylabel("Count")
    axis.grid(alpha=0.25)

plt.tight_layout()
plt.show()



### **3.3. Distribution Comparison**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_A/image_03_03.jpg?v=1783905419" width="250">



>* Compare transformations beyond histogram appearance
>* Preserve predictive and interpretive information

>* Dense features allow more transformation options
>* Sparse features need zeros preserved

>* Match transformations to feature shape and sparsity
>* Check zeros, memory, reliability, and interpretability



# <font color="#418FDE" size="6.5" uppercase>**Scaling Distributions**</font>


In this lecture, you learned to:
- Apply standard, min-max, max-absolute, robust, and sample normalization transformations. 
- Compare quantile and power transformations for skewed numerical features. 
- Choose scaling strategies that respect sparse and dense data constraints. 

In the next Lecture (Lecture B), we will go over 'Expansion Binning'